<a href="https://colab.research.google.com/github/AHNK-05/Python_trading_system/blob/main/algo_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import math
import requests

import os
import io

In [ ]:

# Official NSE Indices constituent files (tried in order).
CONSTITUENT_URLS = [
    "https://www.niftyindices.com/IndexConstituent/ind_nifty50list.csv",
    "https://archives.nseindia.com/content/indices/ind_nifty50list.csv",
    "https://nsearchives.nseindia.com/content/indices/ind_nifty50list.csv",
]

In [ ]:

# NSE blocks requests that don't look like they come from a browser.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept": "text/csv,*/*",
}

SYMBOLS_CACHE = "nifty50_symbols.csv"


In [ ]:

def get_nifty50_symbols(use_cache_on_failure: bool = True) -> list[str]:
    """Return the current Nifty 50 NSE symbols.

    Downloads the official list from NSE Indices and saves a local copy.
    If every download fails, falls back to the last saved copy.
    """
    last_error = None
    for url in CONSTITUENT_URLS:
        try:
            response = requests.get(url, headers=HEADERS, timeout=20)
            response.raise_for_status()
            df = pd.read_csv(io.StringIO(response.text))
            df.columns = [c.strip() for c in df.columns]
            symbols = df["Symbol"].str.strip().tolist()
            if len(symbols) != 50:
                print(f"Warning: expected 50 symbols, got {len(symbols)}")
            df.to_csv(SYMBOLS_CACHE, index=False)  # keep a copy for offline use
            return symbols
        except Exception as exc:  # try the next URL
            last_error = exc

    if use_cache_on_failure and os.path.exists(SYMBOLS_CACHE):
        print(f"Could not download the list ({last_error}). Using cached {SYMBOLS_CACHE}.")
        return pd.read_csv(SYMBOLS_CACHE)["Symbol"].str.strip().tolist()

    raise RuntimeError(f"Could not fetch the Nifty 50 list: {last_error}")

In [ ]:
symbols = get_nifty50_symbols()
print(len(symbols))   # should be 50
print(symbols)

50
['ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', 'ASIANPAINT', 'AXISBANK', 'BSE', 'BAJAJ-AUTO', 'BAJFINANCE', 'BAJAJFINSV', 'BEL', 'BHARTIARTL', 'CIPLA', 'COALINDIA', 'DRREDDY', 'EICHERMOT', 'ETERNAL', 'GRASIM', 'HCLTECH', 'HDFCBANK', 'HDFCLIFE', 'HINDALCO', 'HINDUNILVR', 'ICICIBANK', 'ITC', 'INFY', 'INDIGO', 'JSWSTEEL', 'JIOFIN', 'KOTAKBANK', 'LT', 'M&M', 'MARUTI', 'MAXHEALTH', 'NTPC', 'NESTLEIND', 'ONGC', 'POWERGRID', 'RELIANCE', 'SBILIFE', 'SHRIRAMFIN', 'SBIN', 'SUNPHARMA', 'TCS', 'TATACONSUM', 'TMPV', 'TATASTEEL', 'TECHM', 'TITAN', 'TRENT', 'ULTRACEMCO']


In [8]:
START = "2020-01-01"          # momentum (e.g. 12-1) needs >= ~13 months; more is better for backtests
END = None                    # None = up to the latest available day
OUT_CSV = "nifty50_prices.csv"

In [9]:
def to_yf_ticker(symbol: str) -> str:
    """'RELIANCE' -> 'RELIANCE.NS' (safe if '.NS' is already present)."""
    s = symbol.strip().upper()
    return s if s.endswith(".NS") else f"{s}.NS"

In [10]:
def download_close(tickers, start, end):
    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,   # Close is adjusted for splits/dividends, which is what momentum needs
        progress=False,
        threads=True,
    )
    close = raw["Close"]
    if isinstance(close, pd.Series):          # single-ticker edge case
        close = close.to_frame(tickers[0])
    return close

In [11]:
def fetch_nifty50_prices(symbols, start=START, end=END, out_csv=OUT_CSV):
    tickers = [to_yf_ticker(s) for s in symbols]

    close = download_close(tickers, start, end)

    # yfinance silently returns all-NaN columns for tickers it couldn't fetch -> retry those once
    failed = [t for t in tickers if t not in close.columns or close[t].isna().all()]
    if failed:
        print(f"Retrying {len(failed)} ticker(s): {failed}")
        retry = download_close(failed, start, end)
        for t in failed:
            if t in retry.columns and not retry[t].isna().all():
                close[t] = retry[t]
        failed = [t for t in failed if t not in close.columns or close[t].isna().all()]

    if failed:
        print(f"WARNING: no data for {failed} (renamed/demerged/delisted? check the ticker on Yahoo Finance)")
        close = close.drop(columns=[t for t in failed if t in close.columns])

    # Tidy: tz-naive date index, plain NSE symbols as columns, original symbol order
    close.index = pd.to_datetime(close.index).tz_localize(None)
    close.index.name = "Date"
    close.columns = [c.removesuffix(".NS") for c in close.columns]
    ordered = [to_yf_ticker(s).removesuffix(".NS") for s in symbols]
    close = close[[c for c in ordered if c in close.columns]].sort_index()
    close = close.dropna(how="all")           # drop days with no data at all

    close.to_csv(out_csv)
    print(f"Saved {close.shape[0]} rows x {close.shape[1]} symbols to {out_csv}")
    print(f"Range: {close.index.min().date()} -> {close.index.max().date()}")
    return close

In [13]:
# --- run ---
symbols = get_nifty50_symbols()
prices = fetch_nifty50_prices(symbols)
prices.tail()

Saved 1707 rows x 50 symbols to nifty50_prices.csv
Range: 2020-01-01 -> 2026-11-23


,ADANIENT,ADANIPORTS,APOLLOHOSP,ASIANPAINT,AXISBANK,BSE,BAJAJ-AUTO,BAJFINANCE,BAJAJFINSV,BEL,...,SBIN,SUNPHARMA,TCS,TATACONSUM,TMPV,TATASTEEL,TECHM,TITAN,TRENT,ULTRACEMCO
Date,,,,,,,,,,,,,,,,,,,,,
2026-11-17,2745.100098,1917.000000,8712.5,2625.000000,1369.000000,3864.899902,10770.0,1121.099976,1870.800049,399.549988,...,1108.599976,1870.000000,2406.600098,1112.400024,259.200012,181.500000,1762.000000,4908.000000,3176.300049,12290.0
2026-11-18,2684.199951,1902.599976,8821.0,2580.300049,1360.000000,3755.699951,10570.0,1093.849976,1856.199951,395.799988,...,1135.000000,1871.500000,2379.500000,1130.900024,252.550003,179.000000,1750.000000,4648.100098,3190.000000,12428.0
2026-11-19,2703.100098,1917.099976,8738.0,2595.000000,1380.800049,3765.000000,10817.0,1092.849976,1885.000000,404.200012,...,1156.000000,1900.000000,2342.699951,1144.000000,263.500000,180.889999,1723.000000,4700.000000,3225.000000,12560.0
2026-11-20,2709.000000,1917.000000,9200.0,2608.300049,1446.000000,3781.000000,10765.0,1122.000000,1892.000000,402.000000,...,1165.099976,1884.900024,2376.199951,1155.900024,266.350006,178.000000,1746.199951,4620.000000,3256.100098,12939.0
2026-11-23,2677.899902,1919.000000,9257.0,2580.399902,1388.599976,3757.000000,10643.0,1095.500000,1880.099976,395.799988,...,1132.650024,1860.000000,2460.000000,1133.000000,259.000000,163.000000,1777.500000,4680.000000,3208.000000,12650.0
